In [ ]:
import pandas as pd
from src.ingestion.sidra_mapping import SidraVariables

path = "../data/raw/pam_parana_milho_raw.json"
df_raw = pd.read_json(path)

df = pd.read_json(path).iloc[1:]
# df = pd.read_json(path)
df[["D2N", "MN"]].value_counts()

In [ ]:

import numpy as np
rename_map = {"D1N": "ano", "D2C": "variavel_codigo", "D3C": "municipio_codigo", "D3N": "municipio_nome", "V": "valor"}
df_2 = df[rename_map.keys()].rename(columns=rename_map)
df_2.head()

df_2['valor'] = df_2['valor'].replace(
    {"-": "0", 
    "..": np.nan, 
    "...": np.nan,
    "X": np.nan,
    "x": np.nan,
    }
    ) 
df_2['valor'] = pd.to_numeric(df_2['valor'], errors='coerce') #Coerce forces non-convertible values into Nan
df_2['ano'] = df_2['ano'].astype(int)
df_2['municipio_codigo'] = df_2['municipio_codigo'].astype(int)

In [ ]:
df_2.info()

In [ ]:
df_pivot = df_2.pivot_table(
    index=["municipio_codigo", "municipio_nome", "ano"], columns="variavel_codigo", values="valor", aggfunc="first"
).reset_index()

variables_mapping = {v.value: v.name.lower() for v in SidraVariables}
df_final = df_pivot.rename(columns=variables_mapping)
df_final

In [ ]:
import pygwalker as pyg

gwalker = pyg.walk(df_final)

In [ ]:
from src.ingestion.sidra_mapping import SidraVariables, SidraCrops, SidraLocality
locality = { v.value: v.name.lower() for v in SidraLocality}
sidracrops = { v.value: v.name.lower() for v in SidraCrops}
locality_value = locality.values()
locality_value

locality_name = SidraLocality.PARANA.name.lower()


In [ ]:
from pathlib import Path
import pandas as pd
from src.ingestion.sidra_mapping import SidraVariables, SidraCrops, SidraLocality
import numpy as np

variables_mapping = { v.value: v.name.lower() for v in SidraVariables}
locality_name = SidraLocality.PARANA.name.lower()

raw_dir = "../data/raw/"
dfs_all = []

for crop in SidraCrops:

    crop_name = crop.name.lower()
    file_path = Path(raw_dir) / f"pam_{locality_name}_{crop_name}_raw.json" 

    df = pd.read_json(file_path).iloc[1:]

    rename_map = {"D1N": "ano", "D2C": "variavel_codigo", "D3C": "municipio_codigo", "D3N": "municipio_nome", "V": "valor"}
    df_2 = df[rename_map.keys()].rename(columns=rename_map)

    df_2['valor'] = df_2['valor'].replace(
        {"-": "0", 
        "..": np.nan, 
        "...": np.nan,
        "X": np.nan,
        "x": np.nan,
        }
        ) 
    df_2['valor'] = pd.to_numeric(df_2['valor'], errors='coerce') #Coerce forces non-convertible values into Nan
    df_2['ano'] = df_2['ano'].astype(int)
    df_2['municipio_codigo'] = df_2['municipio_codigo'].astype(int)

    df_pivot = df_2.pivot_table(
        index=["municipio_codigo", "municipio_nome", "ano"], columns="variavel_codigo", values="valor", aggfunc="first"
    ).reset_index()

    df_final = df_pivot.rename(columns=variables_mapping)
    df_final['produto'] = crop_name

    dfs_all.append(df_final)

df_concat = pd.concat(dfs_all, ignore_index=True)
df_concat

In [ ]:
import pygwalker as pyg

gwalker = pyg.walk(df_concat)